# Football AI Engineering — Modular Pipeline Notebook

This notebook provides a clean, Google Colab–compatible interface to the refactored `src/` package.

**Workflow:**
1. Environment setup & match configuration (`configs/norway_wales.yaml` or `configs/chelsea_sunderland.yaml`)
2. Dataset preparation, crop quality filtering & automated label auditing
3. Multi-feature & multi-classifier benchmarking with Unknown distance calibration
4. Persistent player tracking (`ByteTrack` vs `BoT-SORT`) & confidence-weighted temporal stabilization
5. Detection evaluation (IoU matching & confidence sweep) & consolidated analytics report

In [ ]:
# Install dependencies when running in Google Colab
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install ultralytics scikit-learn opencv-python matplotlib pyyaml scipy joblib lap

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from src.config import load_match_config
from src.pipeline.runner import prepare_dataset_for_match, run_match_pipeline

# Choose match config: 'configs/norway_wales.yaml' or 'configs/chelsea_sunderland.yaml'
CONFIG_PATH = PROJECT_ROOT / "configs" / "norway_wales.yaml"
match_cfg = load_match_config(CONFIG_PATH)

video_path = match_cfg.resolve_video_path(PROJECT_ROOT)
if not video_path.is_file() and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        match_cfg.video_path = next(iter(uploaded.keys()))

print("Match:", match_cfg.match_name)
print("Resolved video:", match_cfg.resolve_video_path(PROJECT_ROOT))
print("Label mapping:", match_cfg.label_mapping)

In [ ]:
# Execute the complete end-to-end pipeline
results = run_match_pipeline(
    config=match_cfg,
    base_dir=PROJECT_ROOT,
    max_tracking_frames=180,
    compare_trackers=True,
    tracker_comparison_frames=90,
)

print("Experiment output directory:", results["output_dir"])
print("Held-out classification accuracy:", f"{results['classification_summary']['overall_accuracy']:.2%}")
print("Macro F1:", f"{results['classification_summary']['macro_f1']:.2%}")
print("Stable-label coverage:", f"{results['tracking_summary']['stable_label_coverage']:.2%}")

In [ ]:
# Display generated diagnostic plots and annotated video
from IPython.display import Image, Video, display

out_dir = Path(results["output_dir"])
for img_name in [
    "confusion_matrix_comparison.png",
    "unknown_threshold_calibration.png",
    "misclassification_gallery.png",
    "detections_timeline.png",
]:
    p = out_dir / img_name
    if p.is_file():
        display(Image(filename=str(p)))

vid_path = Path(results["artifacts"]["annotated_video"])
if vid_path.is_file():
    display(Video(str(vid_path), embed=True, width=850))